# CARMEN — Quickstart

**CARMEN** is a cardiorespiratory foundation model over 10 physiological signals
(ECG, ABP, PPG, CVP, CO2, AWP, ICP, RESP_Impedance, RESP_Flow, PAP).

This notebook shows how to (1) build the model, (2) turn raw waveforms into a
packed batch, (3) extract features, and (4) extract features with pretrained weights.

## 0. Setup

In [ ]:
import sys, pathlib
# Put the repo root on sys.path so `import carmen` works from a clone.
# (Not needed if you ran `pip install -e .`)
_p = pathlib.Path.cwd()
_root = next((q for q in [_p, *_p.parents] if (q / 'carmen' / '__init__.py').exists()), _p)
sys.path.insert(0, str(_root))

import torch
from carmen import (
    CARMEN, ModelConfig, DownstreamModelWrapper, make_batch,
)
print('CARMEN quickstart — torch', torch.__version__)

## 1. Build the model

No checkpoint required — this constructs a randomly initialized model so you can
verify the wiring. It uses the release model's input path (0.25 s patches,
per-patch conditioning, PPG loc/scale gated) at a much smaller width and depth.
Swap in pretrained weights in step 4.

In [ ]:
cfg = ModelConfig(
    d_model=128, num_layers=2, patch_size=25, num_signal_types=10,
    cond_trend_mode='patchls', gate_unitless_cond=True, gated_cond_signal_types=[2],
)
model = CARMEN.from_config(cfg).eval()
n = sum(p.numel() for p in model.parameters())
print(f'{n/1e6:.2f}M params | d_model={cfg.d_model} | patch_size={cfg.patch_size}')

## 2. Prepare input signals

Build a `PackedBatch` from raw 1-D waveforms. CARMEN expects **100 Hz** — resample
your data first. `make_batch` accepts `(modality, values)` pairs; with
`collate_mode='any_variate'` the modalities of one patient are grouped so the
encoder can attend across them.

In [ ]:
t = torch.linspace(0, 30, 3000)  # 30 s @ 100 Hz (~72 bpm)
ecg = torch.sin(2*torch.pi*1.2*t)
ppg = torch.sin(2*torch.pi*1.2*t - 0.6)
abp = 80 + 30*torch.sin(2*torch.pi*1.2*t - 0.3)
batch = make_batch([('ecg', ecg), ('ppg', ppg), ('abp', abp)], patch_size=cfg.patch_size)
print('packed values:', tuple(batch.values.shape), '| signal_types:', batch.signal_types.tolist())

## 3. Extract features

`extract_features` runs the bidirectional encoder and returns per-patch
embeddings; we mean-pool over the valid patches to get one vector per patient.

In [ ]:
with torch.no_grad():
    feats = model.extract_features(batch)
enc = feats['encoded']
m = feats['patch_mask'].unsqueeze(-1).float()
pooled = (enc * m).sum(1) / m.sum(1).clamp(min=1.0)
print('encoded:', tuple(enc.shape), '-> pooled features:', tuple(pooled.shape))

## 4. Load pretrained weights (optional)

Download a checkpoint (see `checkpoints/README.md`) and set `CKPT`. The checkpoint
embeds its `ModelConfig`, so the architecture self-restores.

In [ ]:
import os
CKPT = 'checkpoints/carmen-base.pt'   # <- set to your downloaded checkpoint
wrapper = None
if os.path.exists(CKPT):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    wrapper = DownstreamModelWrapper(CKPT, device=device)
    b = make_batch([('ecg', ecg), ('ppg', ppg), ('abp', abp)], patch_size=wrapper.patch_size)
    print('pretrained features:', tuple(wrapper.extract_features(b).shape))
else:
    print(f'No checkpoint at {CKPT} — skipping (see checkpoints/README.md).')